# Fine-tune Qwen2.5-1.5B-Instruct bằng Unsloth (LoRA) trên VlogQA
Notebook này hướng dẫn fine-tune mô hình **Qwen2.5-1.5B-Instruct** với **LoRA** qua thư viện **Unsloth** cho tập dữ liệu **VlogQA** (đọc hiểu hội thoại tiếng Việt).

VlogQA là dataset **Extractive QA** (dạng SQuAD): mô hình cần trích xuất một đoạn văn bản ngắn từ context làm câu trả lời.

**Evaluation metrics:** Exact Match (EM) và F1-score.

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo

## 1. Load Model và Tokenizer với Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1536  # Window 3000 ký tự -> ~1354 tokens max (P99 toàn dataset)
dtype = None           # None để auto-detect (Float16 cho T4/V100, BFloat16 cho Ampere+)
load_in_4bit = True    # Dùng quantization 4-bit để tiết kiệm VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Tải model thành công!")

## 2. Cấu hình LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # Rank của LoRA (16 là cân bằng tốt)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,                 # Scaling factor (thường = 2 * r)
    lora_dropout = 0.05,             # Dropout nhẹ để tránh overfitting
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm VRAM 30%
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Cấu hình LoRA thành công!")

## 3. Chuẩn bị Dataset VlogQA

In [ ]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                # Lấy câu trả lời đầu tiên (có thể có nhiều annotators)
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""  # Bỏ qua câu không có đáp án
                if answer:  # Chỉ lấy mẫu có đáp án
                    samples.append({
                        "context": context,
                        "question": question,
                        "answer": answer,
                        "id": qa.get("id", ""),
                    })
    return samples

# Đường dẫn dữ liệu (điều chỉnh nếu cần)
TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"
TEST_PATH  = "test.json"

train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")
print(f"Ví dụ mẫu đầu tiên:")
print(f"  Context (100 ký tự đầu): {train_samples[0]['context'][:100]}")
print(f"  Question: {train_samples[0]['question']}")
print(f"  Answer: {train_samples[0]['answer']}")

In [ ]:
# ==========================================
# Tạo prompt template cho Extractive QA
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là một trợ lý AI thông minh, giỏi tiếng Việt. "
    "Nhiệm vụ của bạn là đọc đoạn văn và trả lời câu hỏi "
    "bằng cách trích xuất chính xác đoạn văn bản ngắn nhất từ ngữ cảnh được cung cấp."
)

def format_prompt_train(context, question, answer, tokenizer):
    """Tạo prompt đầy đủ (cả input lẫn output) để huấn luyện."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "\n                f"đúng cụm từ xuất hiện trong đoạn văn.\\n\\n"
                f"Đoạn văn:\\n{context}\\n\\n"
                f"Câu hỏi: {question}\\n\\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
        {"role": "assistant", "content": answer},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return text

def format_prompt_inference(context, question, tokenizer):
    """Tạo prompt inference (chỉ input, không có answer)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "\n                f"đúng cụm từ xuất hiện trong đoạn văn.\\n\\n"
                f"Đoạn văn:\\n{context}\\n\\n"
                f"Câu hỏi: {question}\\n\\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text

# Tạo dataset HuggingFace
def prepare_hf_dataset(samples, tokenizer):
    formatted = []
    for s in samples:
        text = format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

dataset = prepare_hf_dataset(train_samples, tokenizer)
print(f"\\nDataset HuggingFace tạo thành công: {len(dataset)} mẫu")
print(f"\n--- Ví dụ prompt đầu tiên ---\n{dataset[0]['text'][:800]}")


## 4. Cấu hình và Bắt đầu Huấn luyện (Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import sys

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,      # Dùng 1 process để tránh lỗi pickle
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 2 * 4 = 8
        warmup_steps = 10,
        num_train_epochs = 3,              # Train 3 epochs
        # max_steps = 200,                 # Hoặc giới hạn số bước để test nhanh
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",      # Cosine decay phù hợp hơn cho QA
        seed = 3407,
        output_dir = "outputs_vlogqa",
        save_strategy = "epoch",           # Lưu checkpoint mỗi epoch
        report_to = "none",
    ),
)

# Fix sys.modules để tránh lỗi pickle với unsloth
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Bắt đầu train!
print("\nBắt đầu fine-tuning...")
trainer_stats = trainer.train()

## 5. Kiểm tra Inference nhanh sau khi huấn luyện

In [ ]:
# Kích hoạt chế độ sinh token nhanh của Unsloth (2x tốc độ)
FastLanguageModel.for_inference(model)

# Thử nghiệm với mẫu đầu tiên trong tập train
sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 64,
    use_cache = True,
    do_sample = False,    # Greedy decoding cho QA
    temperature = 1.0,
)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi : {sample['question']}")
print(f"Đáp án đúng : {sample['answer']}")
print(f"Model trả lời: {response}")

## 6. Lưu Model LoRA

In [ ]:
# Chỉ lưu LoRA weights (nhẹ, vài chục MB)
LORA_SAVE_PATH = "qwen2.5-1.5b-instruct-lora-vlogqa"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình LoRA tại: {LORA_SAVE_PATH}")

# Tùy chọn: Merge LoRA vào model gốc và lưu dạng full weights
# model.save_pretrained_merged("qwen2.5-1.5b-instruct-vlogqa-merged", tokenizer, save_method="merged_16bit")

---
## 7. Đánh giá (Evaluation) trên tập Test
### Metrics: Exact Match (EM) và F1-score

- **Exact Match (EM):** Tỷ lệ câu trả lời của model khớp **hoàn toàn** với đáp án (sau khi chuẩn hóa text).
- **F1-score:** Đo lường overlap từ-vựng giữa dự đoán và đáp án (tính theo cấp độ token).

> **Lưu ý:** Cell này có thể chạy trong một session mới sau khi đã train và lưu model.

In [ ]:
import json
import re
import string
import unicodedata
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from collections import Counter

# ==========================================
# CẤU HÌNH
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH    = "qwen2.5-1.5b-instruct-lora-vlogqa"   # Đường dẫn LoRA đã lưu
TEST_DATA_PATH  = "test.json"

SYSTEM_PROMPT = (
    "Bạn là một trợ lý AI thông minh, giỏi tiếng Việt. "
    "Nhiệm vụ của bạn là đọc đoạn văn và trả lời câu hỏi "
    "bằng cách trích xuất chính xác đoạn văn bản ngắn nhất từ ngữ cảnh được cung cấp."
)

In [ ]:
# ==========================================
# LOAD MODEL ĐÃ FINE-TUNE
# ==========================================
print("Loading Tokenizer và Model...")
tokenizer_eval = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype = torch.float16,
    device_map = "auto"
)
model_eval = PeftModel.from_pretrained(base_model_eval, ADAPTER_PATH)
model_eval.eval()
print("Load model hoàn tất!")

In [ ]:
# ==========================================
# HÀM TIỆN ÍCH: CHUẨN HÓA VĂN BẢN
# ==========================================
def normalize_text(text: str) -> str:
    """
    Chuẩn hóa văn bản để so sánh EM/F1:
    - Chuyển về chữ thường
    - Chuẩn hóa Unicode (NFC)
    - Xóa dấu câu (punctuation)
    - Xóa khoảng trắng thừa
    """
    # Chuẩn hóa Unicode về dạng NFC (quan trọng với tiếng Việt)
    text = unicodedata.normalize("NFC", text)
    # Chuyển về chữ thường
    text = text.lower()
    # Xóa dấu câu
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Xóa khoảng trắng thừa
    text = " ".join(text.split())
    return text


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    """Trả về 1 nếu prediction khớp hoàn toàn với ground_truth (sau chuẩn hóa), 0 nếu không."""
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    """
    Tính F1-score ở cấp độ token (từ) dựa trên overlap.
    Theo chuẩn SQuAD evaluation script.
    """
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


def get_best_score(prediction: str, answers: list) -> tuple:
    """
    Khi có nhiều đáp án ground truth (nhiều annotators),
    lấy điểm cao nhất so với bất kỳ đáp án nào.
    """
    em_scores = [compute_exact_match(prediction, ans["text"]) for ans in answers]
    f1_scores = [compute_f1_token(prediction, ans["text"]) for ans in answers]
    return max(em_scores), max(f1_scores)


print("Các hàm metric đã sẵn sàng!")
# Test nhanh
print("EM test:", compute_exact_match("Maggi", "Maggi"))     # Expected: 1
print("EM test:", compute_exact_match("maggi", "Maggi"))     # Expected: 1 (case-insensitive)
print("F1 test:", compute_f1_token("nước tương Maggi", "Maggi"))  # Partial overlap

In [ ]:
# ==========================================
# HÀM TẠO PROMPT INFERENCE
# ==========================================
def format_prompt_inference_eval(context, question, tokenizer):
    """Tạo prompt cho inference (không có đáp án)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "\n                f"đúng cụm từ xuất hiện trong đoạn văn.\\n\\n"
                f"Đoạn văn:\\n{context}\\n\\n"
                f"Câu hỏi: {question}\\n\\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],   # Giữ toàn bộ danh sách đáp án
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")


In [ ]:
# ==========================================
# CHẠY INFERENCE TRÊN TOÀN BỘ TẬP TEST
# ==========================================
all_em_scores  = []
all_f1_scores  = []
all_predictions = []  # Lưu lại để phân tích sau

print(f"\nBắt đầu đánh giá trên {len(test_samples)} câu hỏi...\n")

for sample in tqdm(test_samples, desc="Evaluating"):
    prompt = format_prompt_inference_eval(
        context   = sample["context"],
        question  = sample["question"],
        tokenizer = tokenizer_eval
    )

    # Tokenize
    inputs = tokenizer_eval(
        prompt,
        return_tensors = "pt",
        truncation = True,
        max_length = 1536,  # Khớp max_seq_length khi train
    ).to(model_eval.device)

    # Generate
    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_new_tokens = 50,           # Đáp án VlogQA thường rất ngắn
            do_sample = False,             # Greedy decoding
            temperature = 1.0,
            pad_token_id = tokenizer_eval.pad_token_id,
            eos_token_id = tokenizer_eval.eos_token_id,
        )

    # Decode phần model sinh ra (bỏ phần prompt)
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    prediction = tokenizer_eval.decode(generated_tokens, skip_special_tokens=True).strip()

    # Lấy dòng đầu tiên nếu model sinh ra nhiều dòng
    prediction = prediction.split("\n")[0].strip()

    # Tính metric với best-of-N (so sánh với tất cả đáp án ground truth)
    em, f1 = get_best_score(prediction, sample["answers"])

    all_em_scores.append(em)
    all_f1_scores.append(f1)
    all_predictions.append({
        "id": sample["id"],
        "question": sample["question"],
        "ground_truth": sample["answers"][0]["text"],
        "prediction": prediction,
        "em": em,
        "f1": f1,
    })

    # In ra console để theo dõi quá trình chạy test
    tqdm.write(f"[{len(all_predictions)}/{len(test_samples)}] Q: {sample['question'][:50]}... | Truth: {sample['answers'][0]['text']} | Pred: {prediction} | EM: {em} | F1: {f1:.4f}")

print("\nHoàn tất inference!")

In [ ]:
# ==========================================
# IN KẾT QUẢ ĐÁNH GIÁ
# ==========================================
total = len(all_em_scores)
exact_match_score = sum(all_em_scores) / total * 100
f1_score_avg      = sum(all_f1_scores) / total * 100

print("\n" + "=" * 55)
print("   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA")
print("   Model: Qwen2.5-1.5B-Instruct + LoRA (Unsloth)")
print("=" * 55)
print(f"  Tổng số câu hỏi test  : {total}")
print(f"  Exact Match (EM)      : {exact_match_score:.2f}%")
print(f"  F1-score (token-level): {f1_score_avg:.2f}%")
print("=" * 55)

# Phân tích chi tiết
correct_em = sum(all_em_scores)
f1_buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
for f1 in all_f1_scores:
    if f1 == 1.0:
        f1_buckets["1.0"] += 1
    elif f1 >= 0.8:
        f1_buckets["0.8-1.0"] += 1
    elif f1 >= 0.5:
        f1_buckets["0.5-0.8"] += 1
    elif f1 >= 0.2:
        f1_buckets["0.2-0.5"] += 1
    else:
        f1_buckets["0-0.2"] += 1

print(f"\n  EM đúng hoàn toàn: {correct_em}/{total} câu")
print("\n  Phân phối F1-score:")
for bucket, count in f1_buckets.items():
    print(f"    F1 = {bucket}: {count} câu ({count/total*100:.1f}%)")

In [ ]:
# ==========================================
# XEM CÁC MẪU SAI ĐỂ PHÂN TÍCH
# ==========================================
wrong_preds = [p for p in all_predictions if p["em"] == 0]
print(f"\nSố câu EM = 0 (sai hoàn toàn): {len(wrong_preds)}")
print("\n--- 5 mẫu sai đầu tiên ---")
for p in wrong_preds[:5]:
    print(f"\n  Q: {p['question']}")
    print(f"  Ground truth: {p['ground_truth']}")
    print(f"  Prediction  : {p['prediction']}")
    print(f"  F1          : {p['f1']:.4f}")

# Lưu kết quả ra file JSON để phân tích sau
with open("eval_results_vlogqa.json", "w", encoding="utf-8") as f:
    json.dump({
        "exact_match": round(exact_match_score, 4),
        "f1_score": round(f1_score_avg, 4),
        "total": total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa.json")